In [ ]:
# для работы с многомерными массивами и научными вычислениями
import numpy as np 
# для работы с итераторами
import itertools
#  для построения графиков
from matplotlib import pylab 
# библиотека для интерактивной визуализации данных.
import plotly.graph_objs as go 
from plotly.offline import iplot,init_notebook_mode
# iplot — функция для отображения интерактивных графиков прямо в блокноте 
# init_notebook_mode() — включает оффлайн-режим Plotly для работы внутри блокнота
init_notebook_mode(connected=True)

#Класс реализующий простую нейронную сеть
class network:
    # в конструкторе создаётся массив весов размером size+1 (доп. вес для смещения)
    def __init__(self,size): # self - ссылка на сам объект-экземпляр класса
        self.weights=np.zeros(size + 1) # задаём нулевые веса в атрибуте объекта weights
    
    # Функция активации (пороговая)
    def activation_function(self, x):
        return np.array(x >= 0, dtype=int)
    
    # Производная пороговой ФА
    def deviation_activation_function(self, x):
        return 1    
     
    # Функция предсказания 
    def predict(self,inputs):
        # добавляется столбец единичных значений к входным данным, чтобы учесть смещение (bias)
        inputs = np.column_stack((np.ones(len(inputs)),inputs))
        # выполняется матричное умножение входных данных на веса.
        # self.activation_function(...) — предсказания для каждого входа.
        return self.activation_function(np.dot(inputs,self.weights.T)) 
    
    # Функция, вычисляющая расстояние Хеминга
    def calculate_E(self,real_outputs,inputs):
        predict_outputs = np.array(self.predict(inputs) >= 0.5, dtype=int) 
        # Сравнивает каждое значение с порогом 0.5, превращая в 1 или 0
        return sum(abs(real_outputs - predict_outputs)) # расстояние Хэмминга
    
    # Функция обучения нейросети с использованием дельта-правила Видроу-Хоффа
    def learn(self, inputs, outputs, full_inputs, full_outputs, show=True):
            
            norm = 0.3 # норма обучения
            if show==True: # вывод результатов на каждом шаге обучения
                out=[]
                k=0
                E=self.calculate_E(full_outputs,full_inputs) # начальная ошибка сети
                
                # пока ошибка не станет нулевой или пока число итераций < 100
                while E != 0 and k < 100: 

                    # СОХРАНЕНИЕ промежуточных результатов 
                    out.append([k, np.round(self.weights, 3), np.array(self.predict(full_inputs) >= 0.5,
                                            dtype=int), E])
                    
                    # вычисляем прогнозируемый выход ФА для данного набора данных на входе
                    predict_outputs = self.predict(inputs) 
                    
                    # ошибка, которая является разницей между целевыми выходами и предсказанными.
                    error = outputs - predict_outputs 
                    
                    # Коррекция весов на основе ошибки 
                    correction=(norm * np.dot(np.column_stack((np.ones(len(inputs)), inputs)).T,
                                    error * self.deviation_activation_function(inputs))) 
                    
                    self.weights += correction # w + delta(w)

                    # Пересчёт ошибки
                    E=self.calculate_E(full_outputs,full_inputs)
        
                    k+=1 

                out.append([k,np.round(self.weights,3), np.array(self.predict(full_inputs) >= 0.5, 
                                                                dtype=int), E])
                return out
            
            else: # то же самое, но без сохранения промежуточных результатов для вывода
                k=0
                E=self.calculate_E(outputs,inputs)
                while E != 0 and k < 100:
                    predict_outputs = self.predict(inputs)
                    error = outputs-predict_outputs
                    correction = (norm * np.dot(np.column_stack((np.ones(len(inputs)), inputs)).T,
                                    error * self.deviation_activation_function(inputs)))
                    self.weights += correction # w + delta(w)
                    E=self.calculate_E(full_outputs,full_inputs)
                    k+=1


In [ ]:
#Задание исходных данных
def bool_func(x):
    return (not(not( x[0] | x[1] )) | x[2] | x[3])

# Инициализация списков входных и выходных данных
full_inputs=[]
full_outputs=[]
for i in range(16):
    full_inputs.append(list(map(int, bin(i)[2:].rjust(4,'0'))))
    # list(map(int, ...)) — перевод строки из '0' и '1' в список целых чисел: [0, 0, 1, 0]
    # bin(i)[2:] - перевод i в двоичный вид без префикса 0b (срез, начиная со второго символа)
    # .rjust(4,'0') - дополнение слева нулями до 4 бит
for i in full_inputs:
    full_outputs.append(bool_func(i))
full_inputs = np.array(full_inputs)
full_outputs = np.array(full_outputs)

#Вывод таблицы истинности
print("Таблица истинности:")
# zip() — соединяет входы и выходы парами.
for i,j in zip(full_inputs, full_outputs): 
    print(i, int(j))


In [ ]:
#Обучение сети на всех данных
netw=network(4) # создаётся объект класса network, который представляет нейронную сеть с 4 входами.
inputs = full_inputs
outputs = full_outputs
out=netw.learn(inputs, outputs, full_inputs,full_outputs,show=True) # обучаем нейронную сеть на всех данных

#Вывод результатов
# создаём таблицу с помощью библиотеки Plotly для визуализации:
trace = go.Table(
    # header — определяет названия колонок в таблице
    header = dict(values=['Номер эпохи k', 
                        'Вектор весов w',
                        'Выходной вектор y',
                        'Суммарная ошибка E']),
    
    # cells — создаём ячейки для данных, используя zip(*output); 
    # каждый столбец данных (эпохи) транспонируется
    cells = dict(values = [*list(zip(*out))]) 
)
data = [trace] 
iplot(data, filename = 'basic_table') # отображает таблицу в Jupyter Notebook с помощью Plotly


In [ ]:
#Построение графика зависимости суммарной ошибки от номера эпохи
#Пороговая функция, все данные

# output - спсиок, где каждая запись - информация о текущей эпохе
# i[-1] - последний элемент записи (кортежа) - значение суммар. ошибки
error=[i[-1] for i in out] # из результата обучения извлекается ошибка
pylab.plot(error,'-ob') # строим график: 'o' — круги, 'b' — синий цвет
pylab.xlabel('k') # номер эпохи
pylab.ylabel('E') # ошибка
pylab.grid(True)
pylab.axis([0, len(error)-1, 0, max(error)+1])
pylab.show()


In [ ]:
#Класс для сети с сигмоидальной функцией активации
class new_network(network):

    # Функция активации (сигмоидальная)
    def activation_function(self, x):
        return 1 / (1 + np.exp(-x))
    
    # Производная ФА
    def deviation_activation_function(self, x):
        x = np.column_stack((np.ones(len(x)), x))
        x = np.dot(x, self.weights.T)
        return self.activation_function(x) * (1 - self.activation_function(x))
    
#обучение сети с сигмоидальной функцией на всех данных
netw=new_network(4)
output_sigma=netw.learn(inputs, outputs, full_inputs,full_outputs,show=True)

#вывод результатов
trace = go.Table(
    header=dict(values=['Номер эпохи k', 
                        'Вектор весов w',
                        'Выходной вектор y',
                        'Суммарная ошибка E']),
    cells=dict(values=[*list(zip(*output_sigma))])
)
data = [trace] 
iplot(data, filename = 'basic_table')


In [ ]:
#Построение графика зависимости суммарной ошибки от номера эпохи
#Сигмоидальная функция, все данные
error=[i[-1] for i in output_sigma]
pylab.plot(error,'-ob')
pylab.xlabel('k')
pylab.ylabel('E')
pylab.grid(True)
pylab.axis([0, len(error)-1, 0, max(error)+1])
pylab.show()


In [ ]:
#Функция для нахождения минимальных входных данных для сети, достаточных для обучения
def find_min_inputs(network):
    for i in range(15,0,-1): # Перебор подмножеств от 15 до 1 элемента
        new_inputs = itertools.combinations(range(16), i)
        for combination in new_inputs: # перебор всех возможных сочетаний индексов входов длиной i 
            inputs=[]
            outputs=[]
            for number in combination:
                # формируем двоичное представление числа (4-битное)
                inputs.append(list(map(int,bin(number)[2:].rjust(4,'0'))))
            for bin_number in inputs:
                # генерируем выходы для поднабора входов через булеву функцию
                outputs.append(bool_func(bin_number))

            netw=network(4) # инициализация сети с 4 входами   
            netw.learn(inputs,outputs,full_inputs,full_outputs) # Обучение сети на выбранном подмножестве

            # Проверка, способна ли сеть корректно предсказывать на всем множестве
            if all(np.array(netw.predict(full_inputs) >= 0.5, dtype=int) == full_outputs): 
                # если сеть обучилась успешно, сохраняем веса и индексы входов
                min_inputs={'weights': netw.weights,'inputs': combination}
                break
    return min_inputs

#Функция для проверки правильности вычисленных минимальных входных данных
def check_min_inputs(network, min_inputs):
    
    inputs=[]
    outputs=[]
    
    # Восстанавливаем входные векторы из сохранённых индексов
    for i in min_inputs['inputs']:
        inputs.append(list(map(int,bin(i)[2:].rjust(4,'0'))))
    
    # Формируем целевые выходы 
    for j in inputs:
        outputs.append(bool_func(j)) 
        
    inputs = np.array(inputs)
    outputs = np.array(outputs)


    print('input sample ',*inputs)
    
    netw = network(4)
    output = netw.learn(inputs,outputs,full_inputs,full_outputs,show=True) # обучение сети на найденном минимальном наборе 

    print('weights:', min_inputs['weights'])
    print('network outputs ', np.array(netw.predict(full_inputs) >= 0.5, dtype=int))
    print('real outputs   ', full_outputs)
    
    #вывод результатов
    trace = go.Table(
        header=dict(values=['Номер эпохи k', 
                            'Вектор весов w',
                            'Выходной вектор y',
                            'Суммарная ошибка E']),
        cells=dict(values=[*list(zip(*output))])
    )
    data = [trace]
    iplot(data, filename = 'basic_table')
    return output


In [ ]:
#Вычисление минимального набора векторов и вывод результатов
#Пороговая функция
min_inputs = find_min_inputs(network)
out = check_min_inputs(network, min_inputs)


In [ ]:
#Построение графика зависимости суммарной ошибки от номера эпохи
#Пороговая функция, минимальный набор данных
error=[i[-1] for i in out]
pylab.plot(error,'-ob')
pylab.xlabel('k')
pylab.ylabel('E')
pylab.grid(True)
pylab.axis([0, len(error)-1, 0, max(error)+1])
pylab.show()


In [ ]:
#Вычисление минимального набора векторов и вывод результатов
#Сигмоидальная функция
min_inputs = find_min_inputs(new_network)
output_sigma = check_min_inputs(new_network, min_inputs)


In [ ]:
#Построение графика зависимости суммарной ошибки от номера эпохи
#Сигмоидальная функция, минимальный набор данных
error=[i[-1] for i in output_sigma]
pylab.plot(error,'-ob')
pylab.xlabel('k')
pylab.ylabel('E')
pylab.grid(True)
pylab.axis([0, len(error)-1, 0, max(error)+1])
pylab.show()
